devassist_ai.py
================
DevAssist AI — RAG-based Support Troubleshooting Assistant (Simplified)

WHAT THIS PROGRAM DOES
-----------------------
A junior developer gives this program either:
  (A) a typed description of the customer's problem, OR
  (B) a screenshot image of the customer's error

The program then:
  1. Reads the two knowledge base Word documents directly
     (Common_Problems_RAG_Knowledge_Base.docx, Complex_Problems_RAG_Knowledge_Base.docx)
  2. Finds the single closest-matching troubleshooting guide
  3. Prints: Problem Statement, Symptoms, Possible Causes,
     Troubleshooting Steps, Solution, and Escalation Guidance
  4. If the match is not confident enough, it does NOT guess —
     it asks the junior developer for more clarification from the customer.

HOW MATCHING WORKS (kept intentionally simple)
------------------------------------------------
TF-IDF (a standard text-similarity technique) turns the customer's problem
text and every knowledge-base guide into vectors, then measures how similar
they are (cosine similarity). No paid API, no internet connection needed.

For images, OCR (pytesseract) reads the visible text out of the screenshot
first, then that extracted text is matched the exact same way.

REQUIREMENTS
------------
    pip install python-docx pytesseract pillow scikit-learn numpy

    Tesseract OCR must also be installed on your machine (this is the actual
    OCR engine; pytesseract is just the Python wrapper around it):
      - Windows : https://github.com/UB-Mannheim/tesseract/wiki
      - Mac     : brew install tesseract
      - Linux   : sudo apt-get install tesseract-ocr

FILES NEEDED IN THE SAME FOLDER AS THIS SCRIPT
------------------------------------------------
    Common_Problems_RAG_Knowledge_Base.docx
    Complex_Problems_RAG_Knowledge_Base.docx

HOW TO RUN
----------
    python devassist_ai.py
        -> opens an interactive menu (type a problem, or point to an image)

    python devassist_ai.py --text "customer cannot log in, invalid credentials"
        -> analyzes text directly from the command line

    python devassist_ai.py --image path/to/screenshot.png
        -> runs OCR on the image, then analyzes the extracted text

## Imports and optional dependencies


In [88]:
import re
import sys
import argparse
import textwrap
from pathlib import Path
from dataclasses import dataclass, field

from docx import Document
from docx.text.paragraph import Paragraph
from docx.table import Table

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Optional OCR imports — only needed if the user analyzes an image.
try:
    import pytesseract
    from PIL import Image

    # Tesseract is installed here on this Windows computer.  The raw string
    # keeps Windows backslashes from being interpreted as escape sequences.
    pytesseract.pytesseract.tesseract_cmd = (
        r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    )
    OCR_AVAILABLE = True
except ImportError:
    OCR_AVAILABLE = False


## Configuration


In [89]:
COMMON_DOC_NAME = "Common_Problems_RAG_Knowledge_Base (1).docx"
COMPLEX_DOC_NAME = "Complex_Problems_RAG_Knowledge_Base (1).docx"

# Notebooks do not define __file__, so start from the current folder.
BASE_DIR = Path.cwd()
notebook_dir = BASE_DIR / "Notebook"
if not (BASE_DIR / COMMON_DOC_NAME).is_file() and notebook_dir.is_dir():
    BASE_DIR = notebook_dir

COMMON_DOC = BASE_DIR / COMMON_DOC_NAME
COMPLEX_DOC = BASE_DIR / COMPLEX_DOC_NAME

# Below this confidence score (0-100), we do NOT trust the match.
# Instead of guessing, the program asks the junior developer for more
# clarification from the customer.
CONFIDENCE_THRESHOLD = 12.0


## Guide data structure


In [90]:
@dataclass
class Guide:
    guide_id: str
    title: str
    category: str = ""
    identifier: str = ""
    problem: str = ""
    visual: list = field(default_factory=list)
    symptoms: list = field(default_factory=list)
    causes: list = field(default_factory=list)
    steps: list = field(default_factory=list)
    decision: str = ""
    solution: str = ""
    escalation: str = ""
    team: str = ""
    related: list = field(default_factory=list)

    def search_text(self) -> str:
        """All the text used to match this guide against a customer problem."""
        # Identifiers and titles are the strongest retrieval signals, so give
        # them more weight than the long explanatory sections.
        return " ".join([
            self.identifier, self.identifier,
            self.title, self.title, self.title,
            self.problem,
            " ".join(self.symptoms),
            " ".join(self.causes),
            " ".join(self.visual),
        ])


## Knowledge-base document loading


In [91]:
# Each guide in the .docx follows this exact layout (produced by the
# DevAssist AI knowledge base template):

SECTION_BAR_PATTERN = re.compile(r"^\s*(\d{1,2})\.\s+(.+)$")
GUIDE_HEADING_PATTERN = re.compile(r"^([A-Z]+-\d+)\s*\|\s*(.+)$")

SECTION_NUMBER_TO_FIELD = {
    1: "problem",
    2: "visual",
    3: "symptoms",
    4: "causes",
    5: "steps",
    6: "decision",
    7: "solution",
    8: None,          # Verification — not needed for junior-facing output
    9: "escalation",
    10: "team",
    11: "related",
}

# Sections that are lists of bullet points rather than one paragraph of text.
LIST_FIELDS = {"visual", "symptoms", "causes", "steps", "related"}


def _clean(text: str) -> str:
    return text.strip()


def parse_docx(file_path: Path) -> list:
    """Parse one knowledge-base .docx file into a list of Guide objects."""
    if not file_path.exists():
        raise FileNotFoundError(
            f"\nKnowledge base file not found:\n  {file_path}\n\n"
            f"Make sure this file is in the same folder as devassist_ai.py."
        )

    doc = Document(file_path)
    items = list(doc.iter_inner_content())  # paragraphs + tables, in order

    guides = []
    current_guide = None
    current_field = None

    for item in items:
        if isinstance(item, Paragraph):
            text = _clean(item.text)
            if not text:
                continue

            # New guide starts, e.g. "API-001  |  API 401 Unauthorized"
            heading_match = GUIDE_HEADING_PATTERN.match(text)
            if heading_match:
                if current_guide is not None:
                    guides.append(current_guide)
                current_guide = Guide(
                    guide_id=heading_match.group(1),
                    title=heading_match.group(2),
                )
                current_field = None
                continue

            if current_guide is None:
                continue  # still inside cover page / TOC / intro text

            # "Primary Identifier: ..." line
            if text.startswith("Primary Identifier:"):
                current_guide.identifier = text.split(":", 1)[1].strip()
                continue

            # Skip the small italic instruction line under section 2
            if text.startswith("If the junior developer uploads"):
                continue

            if current_field is None:
                continue

            # Add this paragraph's text into whichever section we're in
            if current_field in LIST_FIELDS:
                cleaned = re.sub(r"^\[Primary match\]\s*", "", text)
                cleaned = re.sub(r"^\d+\.\s*", "", cleaned)  # strip "1. " numbering
                getattr(current_guide, current_field).append(cleaned)
            else:
                existing = getattr(current_guide, current_field)
                combined = (existing + " " + text).strip() if existing else text
                setattr(current_guide, current_field, combined)

        elif isinstance(item, Table):
            rows = item.rows
            if len(rows) == 0:
                continue

            # 4x2 metadata table -> pull out the Category value
            if len(rows) >= 2 and len(rows[0].cells) == 2:
                for row in rows:
                    label = _clean(row.cells[0].text)
                    value = _clean(row.cells[1].text)
                    if label == "Category" and current_guide is not None:
                        current_guide.category = value
                continue

            # 1x1 section-bar table, e.g. "1. Problem Statement"
            if len(rows) == 1 and len(rows[0].cells) == 1:
                bar_text = _clean(rows[0].cells[0].text)
                match = SECTION_BAR_PATTERN.match(bar_text)
                if match and current_guide is not None:
                    section_number = int(match.group(1))
                    current_field = SECTION_NUMBER_TO_FIELD.get(section_number)
                continue

    if current_guide is not None:
        guides.append(current_guide)

    return guides


def load_knowledge_base() -> list:
    """Load and combine both knowledge base documents."""
    guides = []
    for doc_path in (COMMON_DOC, COMPLEX_DOC):
        guides.extend(parse_docx(doc_path))
    return guides


## TF-IDF matching engine


In [92]:
class DevAssistEngine:
    """Loads the knowledge base once, then matches customer problems against it."""

    def __init__(self):
        self.guides = load_knowledge_base()
        if not self.guides:
            raise RuntimeError("No guides were loaded — check the knowledge base files.")
        self._build_index()

    def _build_index(self):
        corpus = [g.search_text() for g in self.guides]
        self.vectorizer = TfidfVectorizer(
            stop_words="english", ngram_range=(1, 2), sublinear_tf=True
        )
        self.doc_matrix = self.vectorizer.fit_transform(corpus)

    def match(self, problem_text: str, top_k: int = 3) -> list:
        """
        Compare problem_text against every guide.
        Returns a list of (Guide, confidence_score) tuples, best first.
        """
        query_text = self._enrich_screenshot_query(problem_text)
        query_vec = self.vectorizer.transform([query_text])
        scores = cosine_similarity(query_vec, self.doc_matrix)[0]
        ranked_idx = scores.argsort()[::-1][:top_k]
        return [(self.guides[i], round(float(scores[i]) * 100, 1)) for i in ranked_idx]

    @staticmethod
    def _enrich_screenshot_query(problem_text: str) -> str:
        """Add explicit meaning when one screenshot shows both success and failure."""
        text = problem_text or ""
        lower = text.lower()
        has_notification = "notification" in lower
        has_failure = any(word in lower for word in ("error", "unable", "failed"))
        has_success = any(phrase in lower for phrase in (
            "new notification", "new message", "notification sent"
        ))
        if has_notification and has_failure and has_success:
            return (text + " Some users receive notifications while other users do not. "
                    "Compare notification preferences and user settings.")
        return text


## Screenshot OCR support


In [93]:
# ============================================================
# FAST IMAGE OCR
# ============================================================

from PIL import Image, ImageOps
import pytesseract
import time

MAX_OCR_WIDTH = 1280


def prepare_image_for_ocr(image_path):
    """Resize and simplify the screenshot before OCR."""

    image = Image.open(image_path).convert("RGB")
    original_size = image.size

    # Resize only if the screenshot is wider than 1280 px.
    if image.width > MAX_OCR_WIDTH:
        new_width = MAX_OCR_WIDTH
        new_height = int(
            image.height * new_width / image.width
        )

        image = image.resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS
        )

    # OCR does not need the original colours.
    image = ImageOps.grayscale(image)

    # Improve contrast for small UI/error text.
    image = ImageOps.autocontrast(image)

    return image, original_size


def extract_text_from_image_fast(image_path):
    """Fast OCR for Postman, browser, terminal and error screenshots."""

    start = time.perf_counter()

    image, original_size = prepare_image_for_ocr(image_path)

    print(f"Original image: {original_size[0]} x {original_size[1]}")
    print(f"OCR image: {image.width} x {image.height}")

    # PSM 6 is suitable for screenshots containing blocks of text.
    text = pytesseract.image_to_string(
        image,
        config="--oem 3 --psm 6"
    ).strip()

    elapsed = time.perf_counter() - start
    print(f"OCR time: {elapsed:.2f} seconds")

    return text


def print_extracted_text(extracted_text: str, show: bool = False):
    """Show OCR text only when the user explicitly requests it."""
    if show:
        print("\nText extracted from the image:")
        print(extracted_text if extracted_text else "(no readable text found)")
    else:
        print("\nExtracted screenshot text is hidden.")


## Analysis and results display


In [94]:
def analyze(engine: DevAssistEngine, problem_text: str) -> dict:
    """
    Core decision logic:
      - If no usable text, ask for clarification.
      - If the best match's confidence is below the threshold, ask for
        clarification instead of guessing.
      - Otherwise, return the matched guide.
    """
    problem_text = (problem_text or "").strip()

    if not problem_text:
        return {
            "status": "NEEDS_CLARIFICATION",
            "message": (
                "No usable problem description was provided. Please ask the "
                "customer to describe what they were doing, what they expected "
                "to happen, and what actually happened (including any exact "
                "error text)."
            ),
        }

    matches = engine.match(problem_text, top_k=3)
    best_guide, best_score = matches[0]

    if best_score < CONFIDENCE_THRESHOLD:
        return {
            "status": "NEEDS_CLARIFICATION",
            "message": (
                "The description provided doesn't confidently match any guide "
                "in the knowledge base (highest confidence was only "
                f"{best_score}%). Before escalating, ask the customer for more "
                "detail: the exact error message or code shown, which feature "
                "or page they were using, and a screenshot if possible."
            ),
            "closest_guesses": matches,
        }

    return {
        "status": "MATCH_FOUND",
        "guide": best_guide,
        "confidence": best_score,
        "other_matches": matches[1:],
    }


def print_result(result: dict):
    print()
    if result["status"] == "NEEDS_CLARIFICATION":
        print("=" * 70)
        print("MORE INFORMATION NEEDED")
        print("=" * 70)
        print(result["message"])
        if "closest_guesses" in result:
            print("\nClosest (low-confidence) guesses, for reference only:")
            for guide, score in result["closest_guesses"]:
                print(f"  - {guide.guide_id} | {guide.title} ({score}% confidence)")
        print("=" * 70)
        return

    guide = result["guide"]
    print("=" * 70)
    print(f"MATCHED GUIDE: {guide.guide_id}  |  {guide.title}")
    print(f"Category: {guide.category}   |   Confidence: {result['confidence']}%")
    print("=" * 70)

    sections = [
        ("Problem statement", guide.problem),
        ("Symptoms", _format_bullets(guide.symptoms)),
        ("Possible causes", _format_bullets(guide.causes)),
        ("Troubleshooting steps", _format_numbered_steps(guide.steps)),
        ("Solution", guide.solution),
        ("Escalation guidance", guide.escalation),
        ("Escalate to", guide.team),
    ]
    print_section_table(sections)

    if result["other_matches"]:
        print("\nOther possible matches (lower confidence):")
        for g, score in result["other_matches"]:
            print(f"  - {g.guide_id} | {g.title} ({score}%)")

    print("=" * 70)


def _format_bullets(items: list) -> str:
    """Format a list for the Details column."""
    return "\n".join(f"• {item}" for item in items) or "Not specified"


def _format_numbered_steps(steps: list) -> str:
    """Format troubleshooting steps for the Details column."""
    return "\n".join(f"{number}. {step}" for number, step in enumerate(steps, 1)) or "Not specified"


def print_section_table(sections: list):
    """Print the matched guide in clear Section and Details columns."""
    section_width = 24
    details_width = 72
    border = "+" + "-" * (section_width + 2) + "+" + "-" * (details_width + 2) + "+"

    print("\n" + border)
    print(f"| {'SECTION':<{section_width}} | {'DETAILS':<{details_width}} |")
    print(border)

    for section_name, details in sections:
        detail_lines = []
        for paragraph in (details or "Not specified").splitlines() or ["Not specified"]:
            detail_lines.extend(textwrap.wrap(paragraph, width=details_width) or [""])

        section_lines = textwrap.wrap(section_name, width=section_width) or [""]
        row_height = max(len(section_lines), len(detail_lines))
        for index in range(row_height):
            left = section_lines[index] if index < len(section_lines) else ""
            right = detail_lines[index] if index < len(detail_lines) else ""
            print(f"| {left:<{section_width}} | {right:<{details_width}} |")
        print(border)


## Interactive terminal menu


In [95]:
def interactive_mode(engine: DevAssistEngine):
    print("\nDEVASSIST AI — TROUBLESHOOTING ASSISTANT")
    print("1. Describe the customer's problem (text)")
    print("2. Analyze an error screenshot (image)")
    choice = input("\nChoose 1 or 2: ").strip()

    if choice == "1":
        problem_text = input("\nDescribe the customer's problem: ").strip()
        result = analyze(engine, problem_text)
        print_result(result)

    elif choice == "2":
        image_path = input("\nPath to the screenshot image: ").strip()
        try:
            extracted_text = extract_text_from_image_fast(image_path)
        except Exception as e:
            print(f"\nCould not read the image: {e}")
            return

        reveal = input("Show extracted screenshot text? [y/N]: ").strip().lower() in {"y", "yes"}
        print_extracted_text(extracted_text, show=reveal)

        result = analyze(engine, extracted_text)
        print_result(result)

    else:
        print("Please choose 1 or 2.")


## Command-line entry point


In [96]:
def main(argv=None):
    parser = argparse.ArgumentParser(description="DevAssist AI - RAG Support Troubleshooting Assistant")
    parser.add_argument("--text", help="Analyze a customer problem description directly.")
    parser.add_argument("--image", help="Analyze a screenshot image of the customer's error.")
    parser.add_argument(
        "--show-extracted-text",
        action="store_true",
        help="Show OCR text for --image after redacting credential-like values.",
    )
    args = parser.parse_args(argv)

    print("Loading knowledge base...")
    engine = DevAssistEngine()
    print(f"Loaded {len(engine.guides)} guides.\n")

    if args.text:
        result = analyze(engine, args.text)
        print_result(result)
    elif args.image:
        try:
            extracted_text = extract_text_from_image_fast(args.image)
        except Exception as e:
            print(f"Could not read the image: {e}")
            sys.exit(1)
        print_extracted_text(extracted_text, show=args.show_extracted_text)
        result = analyze(engine, extracted_text)
        print_result(result)
    else:
        interactive_mode(engine)


# Run automatically only when this code is saved and executed as a .py file.
if __name__ == "__main__" and "__file__" in globals():
    main()


## Image Processing Speed Test

In [97]:
# Put your screenshot path here.
IMAGE_PATH = r"D:\DevAssist AI\Notebook\Bad request.jpg"
start = time.perf_counter()

try:
    extracted_text = extract_text_from_image_fast(IMAGE_PATH)

    print("\nExtracted text:")
    print(extracted_text)

    print(
        f"\nTotal image processing time: "
        f"{time.perf_counter() - start:.2f} seconds"
    )

except Exception as error:
    print("Image processing error:", error)


Original image: 1536 x 1024
OCR image: 1280 x 853
OCR time: 0.54 seconds

Extracted text:
POST Create Customer @ + -> No Environment v
POST v __ https://api.example.com/api/v1/customers Sae v | “Send

Params Authorization Headers (9) Body @ Scripts _ Settings Cookies

© none © form-data © x-www-form-urlencoded @ raw © binary © GraphQL JSON v Beautify
tt
? “name": "John Doe",

3} "email": "john@example.com"
4 |}

Body Cookies Headers (8) Test Results 400 Bad Request 120ms - 3288 (@) | [J SaveResponse ve
Pretty Raw Preview Visualize JSON v = mT Q
i
2 "error": "bad_request",

3 “message”: "The request is missing required field: phone",
4 "status": 400
5

Total image processing time: 0.54 seconds


In [98]:
interactive_mode(DevAssistEngine())


DEVASSIST AI — TROUBLESHOOTING ASSISTANT
1. Describe the customer's problem (text)
2. Analyze an error screenshot (image)
Original image: 1536 x 1024
OCR image: 1280 x 853
OCR time: 0.53 seconds

Text extracted from the image:
POST Create Customer @ + -> No Environment v
POST v __ https://api.example.com/api/v1/customers Sae v | “Send

Params Authorization Headers (9) Body @ Scripts _ Settings Cookies

© none © form-data © x-www-form-urlencoded @ raw © binary © GraphQL JSON v Beautify
tt
? “name": "John Doe",

3} "email": "john@example.com"
4 |}

Body Cookies Headers (8) Test Results 400 Bad Request 120ms - 3288 (@) | [J SaveResponse ve
Pretty Raw Preview Visualize JSON v = mT Q
i
2 "error": "bad_request",

3 “message”: "The request is missing required field: phone",
4 "status": 400
5

MORE INFORMATION NEEDED
The description provided doesn't confidently match any guide in the knowledge base (highest confidence was only 11.3%). Before escalating, ask the customer for more detail: the e